In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || echo 'NOT ON PATH'

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(['ollama', 'serve'])
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is up')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not come up')

In [ ]:
!git clone https://github.com/orceal-lab/cot-failure-modes.git
%cd cot-failure-modes
!pip install -q -r requirements.txt

In [ ]:
!ollama pull mistral:7b-instruct-q4_K_M
!ollama pull qwen2.5-coder:7b-instruct-q4_K_M
!ollama pull llama3.1:8b
!ollama pull qwen3:4b

In [ ]:
# Ceiling-effect check: arithmetic_singles_hard.json has the same volume
# structure (N independent facts, summed) as arithmetic_singles.json but
# ~100x larger numbers (3-4 digit arithmetic instead of 2-digit). If
# qwen3:4b's flat ~95-100% curve on the original singles set was a ceiling
# effect from the arithmetic being too easy, this should push it down even
# at low N. If it stays flat here too, the flatness isn't about raw
# arithmetic difficulty.
!python run_experiment.py --problems problems/arithmetic_singles_hard.json --model \
  "ollama:mistral:7b-instruct-q4_K_M" \
  "ollama:qwen2.5-coder:7b-instruct-q4_K_M" \
  "ollama:llama3.1:8b" \
  "ollama:qwen3:4b"

In [ ]:
!python analyze.py results/raw_*_arithmetic_singles_hard_*.json --results-dir results/singles_hard
!python compare_models.py results/singles_hard/analysis_*.csv --results-dir results/singles_hard

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print('Zipped results to /kaggle/working/results.zip')